# Chapter 11 Computational Lab
## Joint Distributions, Independence and Covariance

This notebook accompanies Chapter 11 of *Probability Theory with Python and AI*.

The chapter moves from one-dimensional laws to random vectors and develops the central language for dependence: joint laws, marginals, mutual independence, covariance, correlation, covariance matrices, transformations, order statistics, convolution and the bivariate normal family.

### Learning goals

By the end of the lab you should be able to:

1. work with random vectors and joint cdfs;
2. recover rectangle probabilities and marginal cdfs from a joint cdf;
3. construct and marginalize joint pmfs;
4. integrate joint densities over non-rectangular regions;
5. distinguish marginal densities from the existence of a joint density;
6. apply multivariate LOTUS;
7. distinguish pairwise from mutual independence;
8. use cdf, pmf and density factorization criteria for independence;
9. understand why expectation factorization **characterizes** independence;
10. use Jacobian transformations for random vectors;
11. compute covariance, correlation and covariance matrices;
12. explain why zero covariance does not imply independence in general;
13. verify positive semidefiniteness of covariance matrices;
14. work with multinomial count vectors;
15. derive distributions of maxima, minima and order statistics;
16. use discrete and continuous convolution;
17. construct and analyze a bivariate normal pair;
18. explain why zero correlation implies independence inside the non-degenerate bivariate normal family;
19. simulate Galton's quincunx and separate model assumptions from physical implementation;
20. audit AI-generated claims about joint distributions and dependence.

> **Chapter boundary.** Conditional distributions and conditional expectation are postponed to Chapter 12.


## 0. Setup

The notebook uses NumPy and Matplotlib together with exact finite calculations whenever possible.


In [ ]:
from fractions import Fraction
from itertools import product
from math import comb, factorial, pi, sqrt
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def numerical_integral_1d(y, x):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y, x)
    return np.trapz(y, x)


def joint_pmf_marginals(table):
    table = np.asarray(table, dtype=float)
    return table.sum(axis=1), table.sum(axis=0)


def covariance_from_joint(x_values, y_values, table):
    x_values = np.asarray(x_values, dtype=float)
    y_values = np.asarray(y_values, dtype=float)
    table = np.asarray(table, dtype=float)

    px = table.sum(axis=1)
    py = table.sum(axis=0)

    EX = np.sum(x_values * px)
    EY = np.sum(y_values * py)

    EXY = 0.0
    for i, x in enumerate(x_values):
        for j, y in enumerate(y_values):
            EXY += x*y*table[i, j]

    return EX, EY, EXY, EXY-EX*EY


def binomial_pmf(k, n, p):
    if k < 0 or k > n:
        return 0.0
    return comb(n, k)*(p**k)*((1-p)**(n-k))


def bivariate_normal_sample(N, mu_x, mu_y, sigma_x, sigma_y, rho, seed=2026):
    rng = np.random.default_rng(seed)
    z1 = rng.normal(size=N)
    z2 = rng.normal(size=N)

    X = mu_x + sigma_x*z1
    Y = mu_y + sigma_y*(rho*z1 + math.sqrt(1-rho**2)*z2)

    return X, Y


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))
    for line in latex_lines:
        display(Math(line))
    if note:
        display(Markdown(note))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Joint-distribution tools are ready."
    "</div>"
))


## 1. Random vectors and joint cdfs

For real-valued random variables $X_1,\ldots,X_d$ on the same probability space,

$$
\mathbf X=(X_1,\ldots,X_d)
$$

is a $d$-dimensional random vector.

Its joint cdf is

$$
\boxed{
F_{\mathbf X}(x_1,\ldots,x_d)
=
P(X_1\le x_1,\ldots,X_d\le x_d).
}
$$

For two coordinates we write

$$
F_{X,Y}(x,y).
$$


### Rectangle probabilities

For $a<b$ and $c<d$,

$$
\boxed{
P(a<X\le b,\ c<Y\le d)
=
F(b,d)-F(a,d)-F(b,c)+F(a,c).
}
$$

The four terms are a two-dimensional inclusion--exclusion formula.


In [ ]:
# Independent Bernoulli(1/2) pair.
def coin_joint_cdf(x, y):
    values = [(0,0),(0,1),(1,0),(1,1)]
    return sum(
        0.25
        for a,b in values
        if a <= x and b <= y
    )

a, b, c, d = -0.1, 0.5, -0.1, 0.5

rectangle = (
    coin_joint_cdf(b,d)
    - coin_joint_cdf(a,d)
    - coin_joint_cdf(b,c)
    + coin_joint_cdf(a,c)
)

display(Math(
    r"P(-0.1<X\le0.5,\,-0.1<Y\le0.5)="
    + f"{rectangle:.2f}"
))


### Joint cdf properties

A joint cdf is:

- non-decreasing in every coordinate;
- right-continuous in the coordinatewise sense;
- zero when any one coordinate tends to $-\infty$ with the others fixed;
- one when all coordinates tend to $+\infty$;
- non-negative on every half-open rectangle increment.

The last property is the multidimensional analogue of monotonicity of a one-dimensional cdf.


## 2. Marginal cdfs

Marginal laws are recovered by letting the other coordinates become unrestricted.

For a pair,

$$
\boxed{
F_X(x)=\lim_{y\to\infty}F_{X,Y}(x,y),
}
$$

and

$$
\boxed{
F_Y(y)=\lim_{x\to\infty}F_{X,Y}(x,y).
}
$$


In [ ]:
for x in [0.0, 0.5, 1.0]:
    marginal = coin_joint_cdf(x, 10)
    display(Math(
        r"F_X(" + f"{x:g}" + r")=" + f"{marginal:.2f}"
    ))


## 3. Discrete joint laws and joint pmfs

A joint law is discrete when it is concentrated on a finite or countable subset of $\mathbb R^d$.

For a discrete pair,

$$
\boxed{
p_{X,Y}(x,y)
=
P(X=x,Y=y).
}
$$

Marginalization is summation:

$$
\boxed{
p_X(x)=\sum_y p_{X,Y}(x,y),
\qquad
p_Y(y)=\sum_x p_{X,Y}(x,y).
}
$$


### A dependent discrete pair

Consider

$$
\begin{array}{c|ccc}
 &Y=0&Y=1&Y=2\\
\hline
X=0&1/6&1/6&0\\
X=1&0&1/3&1/3
\end{array}
$$

The marginals are

$$
P(X=0)=\frac13,
\qquad
P(X=1)=\frac23,
$$

and

$$
P(Y=0)=\frac16,
\qquad
P(Y=1)=\frac12,
\qquad
P(Y=2)=\frac13.
$$


In [ ]:
joint_table = np.array([
    [1/6, 1/6, 0],
    [0, 1/3, 1/3],
])

px, py = joint_pmf_marginals(joint_table)

display(Markdown(f"Row sums $p_X$: **{px}**"))
display(Markdown(f"Column sums $p_Y$: **{py}**"))

product_cell = px[1]*py[0]
joint_cell = joint_table[1,0]

display(Math(r"P(X=1,Y=0)=" + f"{joint_cell:.4f}"))
display(Math(r"P(X=1)P(Y=0)=" + f"{product_cell:.4f}"))
display(Markdown(f"Independent: **{np.allclose(joint_table, np.outer(px,py))}**"))


### Marginals do not determine the joint law

Two different joint tables can have exactly the same row and column sums.

Dependence is information about **how values occur together**, and that information is invisible in the marginals alone.


In [ ]:
independent = np.array([
    [0.25, 0.25],
    [0.25, 0.25],
])

dependent = np.array([
    [0.50, 0.00],
    [0.00, 0.50],
])

display(Markdown(
    f"Same row marginals: **{np.allclose(independent.sum(axis=1), dependent.sum(axis=1))}**"
))
display(Markdown(
    f"Same column marginals: **{np.allclose(independent.sum(axis=0), dependent.sum(axis=0))}**"
))
display(Markdown(
    f"Different joint tables: **{not np.allclose(independent, dependent)}**"
))


## 4. Joint densities, regions and marginals

A non-negative Borel function $f_{X,Y}$ is a joint density when

$$
\boxed{
P((X,Y)\in A)
=
\iint_A
f_{X,Y}(x,y)\,dx\,dy
}
$$

for every Borel region $A\subseteq\mathbb R^2$.

In particular,

$$
\iint_{\mathbb R^2}
f_{X,Y}(x,y)\,dx\,dy
=
1.
$$


### Marginal densities

If $(X,Y)$ has joint density $f_{X,Y}$, then

$$
\boxed{
f_X(x)
=
\int_{-\infty}^{\infty}
f_{X,Y}(x,y)\,dy,
}
$$

and

$$
\boxed{
f_Y(y)
=
\int_{-\infty}^{\infty}
f_{X,Y}(x,y)\,dx,
}
$$

with equality almost everywhere.


### Uniform density on a triangle

Let

$$
f_{X,Y}(x,y)
=
\begin{cases}
2,&0<y<x<1,\\
0,&\text{otherwise}.
\end{cases}
$$

The support triangle has area $1/2$, so the total probability is one.

The marginals are

$$
f_X(x)=2x,
\qquad
0<x<1,
$$

and

$$
f_Y(y)=2(1-y),
\qquad
0<y<1.
$$


In [ ]:
grid = np.linspace(0, 1, 500)
Xg, Yg = np.meshgrid(grid, grid)

density = np.where((Yg > 0) & (Yg < Xg) & (Xg < 1), 2.0, 0.0)

fig, ax = plt.subplots(figsize=(5,5))
ax.contourf(Xg, Yg, density, levels=[0,1,2.1], alpha=0.4)
ax.plot([0,1],[0,1])
ax.set_xlim(0,1)
ax.set_ylim(0,1)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Support of the triangular joint density")
plt.show()


### Integrating over a non-rectangular event

For the triangular law,

$$
P(Y<X/2)
=
\int_0^1
\int_0^{x/2}
2\,dy\,dx
=
\frac12.
$$


In [ ]:
x = np.linspace(0,1,10001)
inner_integral = x
probability = numerical_integral_1d(inner_integral, x)

display(Math(
    r"P(Y<X/2)\approx" + f"{probability:.8f}"
))


## 5. Marginal densities do not imply a joint density

Let

$$
X\sim U(0,1)
$$

and define

$$
Y=X.
$$

Both marginals have densities, but

$$
P(Y=X)=1.
$$

The diagonal

$$
D=\{(x,y):y=x\}
$$

has planar area zero.

If a planar joint density existed, every planar null set would have probability zero, contradicting $P((X,Y)\in D)=1$.


In [ ]:
rng = np.random.default_rng(2026)
X = rng.random(2500)
Y = X

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(X, Y, s=7, alpha=0.4)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("A singular pair: all mass lies on the diagonal")
plt.show()


## 6. Multivariate LOTUS

For Borel $g:\mathbb R^d\to\mathbb R$:

- if the joint law is discrete,

$$
\mathbb E[g(\mathbf X)]
=
\sum_{\mathbf x}
g(\mathbf x)p_{\mathbf X}(\mathbf x),
$$

when defined;

- if $\mathbf X$ has joint density $f_{\mathbf X}$,

$$
\boxed{
\mathbb E[g(\mathbf X)]
=
\int_{\mathbb R^d}
g(\mathbf x)f_{\mathbf X}(\mathbf x)\,d\mathbf x.
}
$$

This is the same expectation constructed in Chapter 7, represented through the joint law.


### Mixed moment on the triangle

For the triangular density,

$$
\mathbb E[XY]
=
\int_0^1
\int_0^x
2xy\,dy\,dx
=
\frac14.
$$

Also,

$$
\mathbb E[X]=\frac23,
\qquad
\mathbb E[Y]=\frac13.
$$

Hence

$$
\mathbb E[XY]-\mathbb E[X]\mathbb E[Y]
=
\frac1{36}.
$$


In [ ]:
EX = 2/3
EY = 1/3
EXY = 1/4

display(Math(r"\mathbb E[X]=" + f"{EX:.8f}"))
display(Math(r"\mathbb E[Y]=" + f"{EY:.8f}"))
display(Math(r"\mathbb E[XY]=" + f"{EXY:.8f}"))
display(Math(
    r"\mathbb E[XY]-\mathbb E[X]\mathbb E[Y]="
    + f"{EXY-EX*EY:.8f}"
))


## 7. Mutual independence

Random variables $X_1,\ldots,X_n$ are mutually independent if for every choice of Borel sets $A_1,\ldots,A_n$,

$$
\boxed{
P(X_1\in A_1,\ldots,X_n\in A_n)
=
\prod_{i=1}^n
P(X_i\in A_i).
}
$$

A sequence is mutually independent when every finite subfamily is mutually independent.


### Pairwise versus mutual independence

Pairwise independence asks only that every pair be independent.

Mutual independence is stronger.


### Pairwise independent but not mutually independent

Let $U,V$ be independent Bernoulli$(1/2)$ variables and define

$$
X=U,
\qquad
Y=V,
\qquad
Z=(U+V)\bmod2.
$$

Every pair among $X,Y,Z$ is independent, but

$$
P(X=0,Y=0,Z=0)
=
\frac14,
$$

whereas

$$
P(X=0)P(Y=0)P(Z=0)
=
\frac18.
$$


In [ ]:
outcomes = [(0,0),(0,1),(1,0),(1,1)]

triples = [
    (u, v, (u+v)%2)
    for u,v in outcomes
]

display(Markdown(f"Possible triples: **{triples}**"))

joint000 = sum(
    0.25
    for x,y,z in triples
    if (x,y,z)==(0,0,0)
)

display(Math(r"P(X=0,Y=0,Z=0)=" + f"{joint000:.2f}"))
display(Math(r"P(X=0)P(Y=0)P(Z=0)=0.125"))


### i.i.d.

Variables are **independent and identically distributed** when:

1. they are mutually independent;
2. they have the same probability law.

For an infinite i.i.d. sequence, every finite subfamily must be mutually independent.


## 8. Cdf characterization of mutual independence

Random variables $X_1,\ldots,X_n$ are mutually independent if and only if

$$
\boxed{
F_{X_1,\ldots,X_n}(x_1,\ldots,x_n)
=
\prod_{i=1}^n
F_{X_i}(x_i)
}
$$

for every $(x_1,\ldots,x_n)$.

The converse is an application of the $\pi$--$\lambda$ extension method from Chapter 2: factorization on lower half-lines is extended coordinate by coordinate to all Borel sets.


### Pmf and density factorization

For discrete laws,

$$
\boxed{
p_{X_1,\ldots,X_n}(x_1,\ldots,x_n)
=
\prod_i p_{X_i}(x_i)
}
$$

for all values if and only if the variables are mutually independent.

For variables with marginal densities, mutual independence is equivalent to the existence of a joint density satisfying

$$
\boxed{
f_{X_1,\ldots,X_n}(x_1,\ldots,x_n)
=
\prod_i f_{X_i}(x_i)
}
$$

almost everywhere.


In [ ]:
# Independent uniform coordinates on the unit square.
grid = np.linspace(0,1,200)
Xg, Yg = np.meshgrid(grid, grid)
joint = np.ones_like(Xg)

fig, ax = plt.subplots(figsize=(5,5))
ax.contourf(Xg, Yg, joint, alpha=0.35)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Factorized uniform density on the unit square")
plt.show()


### Separate measurable transformations preserve independence

If $X_1,\ldots,X_n$ are mutually independent and each $g_i$ is Borel measurable, then

$$
g_1(X_1),\ldots,g_n(X_n)
$$

remain mutually independent.

Examples include standardization, squaring one variable, or exponentiating another.


## 9. Expectation factorization characterizes independence

The current chapter uses the stronger equivalence:

$$
\boxed{
X_1,\ldots,X_n
\text{ are mutually independent}
}
$$

if and only if for every bounded Borel $g_1,\ldots,g_n$,

$$
\boxed{
\mathbb E\left[
\prod_{i=1}^n
g_i(X_i)
\right]
=
\prod_{i=1}^n
\mathbb E[g_i(X_i)].
}
$$

The forward direction is first proved for non-negative simple functions, then extended by monotone convergence.

The converse follows by choosing

$$
g_i=\mathbf 1_{A_i}.
$$


### Extensions

Under independence, the same factorization also holds:

- for non-negative Borel functions, with extended values allowed;
- for signed Borel functions when every $g_i(X_i)$ is integrable.

For independent integrable transforms, the product is integrable as well.


In [ ]:
# Independent finite variables, verifying expectation factorization directly.

x_vals = np.array([0,1])
px = np.array([0.4,0.6])

y_vals = np.array([1,3])
py = np.array([0.7,0.3])

g = lambda x: 2*x + 1
h = lambda y: y*y

Eg = np.sum(np.array([g(x) for x in x_vals])*px)
Eh = np.sum(np.array([h(y) for y in y_vals])*py)

Egh = 0.0
for i,x in enumerate(x_vals):
    for j,y in enumerate(y_vals):
        Egh += g(x)*h(y)*px[i]*py[j]

display(Math(r"\mathbb E[g(X)]=" + f"{Eg:.6f}"))
display(Math(r"\mathbb E[h(Y)]=" + f"{Eh:.6f}"))
display(Math(r"\mathbb E[g(X)h(Y)]=" + f"{Egh:.6f}"))
display(Markdown(f"Factorization verified: **{abs(Egh-Eg*Eh) < 1e-12}**"))


## 10. Transformations of random vectors

Suppose $(X,Y)$ has joint density $f_{X,Y}$ and

$$
T(x,y)=(u,v)
$$

is a smooth one-to-one transformation with smooth inverse

$$
S(u,v)=(x,y).
$$

Then

$$
\boxed{
f_{U,V}(u,v)
=
f_{X,Y}(S(u,v))
\left|
\det DS(u,v)
\right|.
}
$$

The inverse Jacobian rescales area.


### Stretching a uniform square

If $X,Y$ are independent $U(0,1)$ variables and

$$
U=2X,
\qquad
V=3Y,
$$

then

$$
X=\frac U2,
\qquad
Y=\frac V3,
$$

and

$$
\left|\det DS\right|
=
\frac16.
$$

Thus $(U,V)$ is uniform on $(0,2)\times(0,3)$ with density $1/6$.


In [ ]:
rng = np.random.default_rng(2026)
X = rng.random(4000)
Y = rng.random(4000)

U = 2*X
V = 3*Y

fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(U,V,s=5,alpha=0.25)
ax.set_xlim(0,2)
ax.set_ylim(0,3)
ax.set_xlabel("U")
ax.set_ylabel("V")
ax.set_title("Uniform square stretched to a rectangle")
plt.show()


### Solved Jacobian transformation

If $X,Y$ are independent $U(0,1)$ and

$$
U=X+Y,
\qquad
V=X-Y,
$$

then

$$
X=\frac{U+V}{2},
\qquad
Y=\frac{U-V}{2},
$$

with

$$
\left|
\det
\frac{\partial(x,y)}{\partial(u,v)}
\right|
=
\frac12.
$$

The support becomes

$$
|v|<u<2-|v|,
\qquad
-1<v<1,
$$

and

$$
f_{U,V}(u,v)=\frac12
$$

on this support.


In [ ]:
u = np.linspace(0,2,500)
v_upper = np.minimum(u,2-u)
v_lower = -v_upper

fig, ax = plt.subplots(figsize=(6,5))
ax.fill_between(u, v_lower, v_upper, alpha=0.3)
ax.plot(u, v_upper)
ax.plot(u, v_lower)
ax.set_xlabel("u")
ax.set_ylabel("v")
ax.set_title("Support after U=X+Y, V=X-Y")
plt.show()


## 11. Covariance

For square-integrable $X,Y$,

$$
\boxed{
\operatorname{Cov}(X,Y)
=
\mathbb E[
(X-\mathbb E[X])
(Y-\mathbb E[Y])
].
}
$$

The computational formula is

$$
\boxed{
\operatorname{Cov}(X,Y)
=
\mathbb E[XY]
-
\mathbb E[X]\mathbb E[Y].
}
$$


### Covariance on the triangle

For the triangular joint density,

$$
\mathbb E[X]=\frac23,
\qquad
\mathbb E[Y]=\frac13,
\qquad
\mathbb E[XY]=\frac14.
$$

Therefore

$$
\boxed{
\operatorname{Cov}(X,Y)=\frac1{36}.
}
$$


In [ ]:
display(Math(
    r"\operatorname{Cov}(X,Y)=" + f"{1/36:.8f}"
))


### Bilinearity

For square-integrable variables,

$$
\boxed{
\operatorname{Cov}
\left(
\sum_i a_iX_i,
\sum_j b_jY_j
\right)
=
\sum_i\sum_j
a_ib_j
\operatorname{Cov}(X_i,Y_j).
}
$$

Consequently,

$$
\boxed{
\operatorname{Var}
\left(
\sum_i a_iX_i
\right)
=
\sum_i a_i^2\operatorname{Var}(X_i)
+
2\sum_{i<j}
a_ia_j\operatorname{Cov}(X_i,X_j).
}
$$


In [ ]:
varX = 4
varY = 9
covXY = -2

var_combo = (2**2)*varX + ((-3)**2)*varY + 2*(2)*(-3)*covXY

display(Math(
    r"\operatorname{Var}(2X-3Y)=" + f"{var_combo:g}"
))


## 12. Independence implies zero covariance

If $X$ and $Y$ are independent and have finite second moments, then

$$
\boxed{
\operatorname{Cov}(X,Y)=0.
}
$$

This follows immediately from expectation factorization:

$$
\mathbb E[XY]
=
\mathbb E[X]\mathbb E[Y].
$$

The converse is false in general.


### Uncorrelated but dependent

Let

$$
P(X=-1)=P(X=0)=P(X=1)=\frac13,
$$

and define

$$
Y=X^2.
$$

By symmetry,

$$
\mathbb E[X]=0,
\qquad
\mathbb E[XY]
=
\mathbb E[X^3]
=
0,
$$

so

$$
\operatorname{Cov}(X,Y)=0.
$$

But $Y$ is completely determined by $X$, so the variables are dependent.


In [ ]:
x = np.array([-1,0,1])
y = x**2

fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(x,y,s=100)
ax.set_xlabel("X")
ax.set_ylabel("Y=X^2")
ax.set_title("Zero correlation can coexist with deterministic dependence")
plt.show()

p = np.array([1/3,1/3,1/3])
EX = np.sum(x*p)
EY = np.sum(y*p)
EXY = np.sum(x*y*p)
cov = EXY-EX*EY

display(Math(r"\operatorname{Cov}(X,Y)=" + f"{cov:.1f}"))


## 13. Correlation coefficient

If both variances are positive and finite,

$$
\boxed{
\rho_{X,Y}
=
\frac{
\operatorname{Cov}(X,Y)
}{
\sqrt{
\operatorname{Var}(X)
\operatorname{Var}(Y)
}
}.
}
$$

Cauchy--Schwarz gives

$$
\boxed{
-1\le\rho_{X,Y}\le1.
}
$$

Moreover,

$$
|\rho_{X,Y}|=1
$$

if and only if

$$
Y=aX+b
$$

almost surely for some $a\ne0$.


In [ ]:
corr_a = widgets.FloatSlider(value=2.0, min=-4, max=4, step=0.25, description="a")
corr_b = widgets.FloatSlider(value=-5.0, min=-10, max=10, step=0.5, description="b")
corr_output = widgets.Output()


def update_perfect_corr(*_):
    with corr_output:
        clear_output(wait=True)

        a = corr_a.value
        b = corr_b.value

        if abs(a) < 1e-12:
            display(Markdown("**Choose nonzero a.**"))
            return

        rng = np.random.default_rng(123)
        X = rng.normal(size=5000)
        Y = a*X+b

        rho = np.corrcoef(X,Y)[0,1]

        display(Math(r"\widehat\rho=" + f"{rho:.8f}"))
        display(Markdown(
            "**Perfect positive correlation**"
            if a > 0
            else "**Perfect negative correlation**"
        ))


for control in (corr_a,corr_b):
    control.observe(update_perfect_corr, names="value")

display(widgets.VBox([
    widgets.HBox([corr_a,corr_b]),
    corr_output,
]))
update_perfect_corr()


## 14. Mean vector and covariance matrix

For

$$
\mathbf X=(X_1,\ldots,X_d)^\top,
$$

the covariance matrix is

$$
\boxed{
\Sigma_{ij}
=
\operatorname{Cov}(X_i,X_j).
}
$$

It is symmetric and positive semidefinite.


### Quadratic-form identity

For every vector $\mathbf a$,

$$
\boxed{
\operatorname{Var}(\mathbf a^\top\mathbf X)
=
\mathbf a^\top
\Sigma
\mathbf a
\ge0.
}
$$

This proves that every covariance matrix is positive semidefinite.


In [ ]:
Sigma_triangle = np.array([
    [1/18, 1/36],
    [1/36, 1/18],
])

eigenvalues = np.linalg.eigvalsh(Sigma_triangle)

display(Markdown(f"Covariance matrix:\n\n`{Sigma_triangle}`"))
display(Markdown(f"Eigenvalues: **{eigenvalues}**"))
display(Markdown(
    f"Positive semidefinite: **{np.all(eigenvalues >= -1e-12)}**"
))


In [ ]:
psd_a = widgets.FloatSlider(value=1.0, min=-5, max=5, step=0.25, description="a")
psd_b = widgets.FloatSlider(value=-2.0, min=-5, max=5, step=0.25, description="b")
psd_output = widgets.Output()


def update_psd(*_):
    with psd_output:
        clear_output(wait=True)

        vec = np.array([psd_a.value, psd_b.value])
        q = float(vec @ Sigma_triangle @ vec)

        display(Math(
            r"\mathbf a^\top\Sigma\mathbf a="
            + f"{q:.8f}"
        ))


for control in (psd_a,psd_b):
    control.observe(update_psd, names="value")

display(widgets.VBox([
    widgets.HBox([psd_a,psd_b]),
    psd_output,
]))
update_psd()


## 15. The multinomial distribution

Suppose $n$ independent categorical trials produce one of $m$ categories with probabilities

$$
p_1,\ldots,p_m,
\qquad
\sum_i p_i=1.
$$

Let $N_i$ be the number of trials in category $i$.

Then

$$
(N_1,\ldots,N_m)
\sim
\operatorname{Multinomial}
(n;p_1,\ldots,p_m),
$$

with pmf

$$
\boxed{
P(N_1=n_1,\ldots,N_m=n_m)
=
\frac{n!}{n_1!\cdots n_m!}
\prod_i p_i^{n_i},
}
$$

when $\sum_i n_i=n$.


### Multinomial moments

For each category,

$$
\boxed{
\mathbb E[N_i]=np_i,
\qquad
\operatorname{Var}(N_i)=np_i(1-p_i).
}
$$

For $i\ne j$,

$$
\boxed{
\operatorname{Cov}(N_i,N_j)
=
-np_ip_j.
}
$$

Different category counts are negatively correlated because one trial cannot belong to two categories simultaneously.


In [ ]:
multi_n = widgets.IntSlider(value=12, min=1, max=100, description="n")
multi_output = widgets.Output()


def update_multinomial(*_):
    with multi_output:
        clear_output(wait=True)

        n = multi_n.value
        p = np.repeat(1/6, 6)

        Sigma = np.empty((6,6))
        for i in range(6):
            for j in range(6):
                if i == j:
                    Sigma[i,j] = n*p[i]*(1-p[i])
                else:
                    Sigma[i,j] = -n*p[i]*p[j]

        display(Math(
            r"\operatorname{Cov}(N_1,N_2)="
            + f"{Sigma[0,1]:.6f}"
        ))
        display(Markdown(
            f"Smallest eigenvalue of covariance matrix: **{np.linalg.eigvalsh(Sigma)[0]:.3e}**"
        ))
        display(Markdown(
            "The near-zero eigenvalue reflects the deterministic identity "
            "$N_1+\\cdots+N_6=n$."
        ))


multi_n.observe(update_multinomial, names="value")
display(widgets.VBox([multi_n,multi_output]))
update_multinomial()


In [ ]:
multi_sim_N = widgets.IntSlider(value=10000, min=500, max=50000, step=500, description="simulations")
multi_sim_output = widgets.Output()


def update_multinomial_sim(*_):
    with multi_sim_output:
        clear_output(wait=True)

        N = multi_sim_N.value
        rng = np.random.default_rng(2026)
        samples = rng.multinomial(12, [1/6]*6, size=N)

        empirical_cov = np.cov(samples, rowvar=False, ddof=0)

        display(Math(
            r"\widehat{\operatorname{Cov}}(N_1,N_2)="
            + f"{empirical_cov[0,1]:.6f}"
        ))
        display(Math(r"\text{theory}=-\frac13"))


multi_sim_N.observe(update_multinomial_sim, names="value")
display(widgets.VBox([multi_sim_N,multi_sim_output]))
update_multinomial_sim()


## 16. Maxima and minima of independent variables

Let $X_1,\ldots,X_n$ be mutually independent with cdfs $F_1,\ldots,F_n$.

For

$$
M=\max_iX_i,
$$

$$
\boxed{
F_M(x)
=
\prod_iF_i(x).
}
$$

For

$$
m=\min_iX_i,
$$

$$
\boxed{
F_m(x)
=
1-\prod_i(1-F_i(x)).
}
$$


### Maximum of i.i.d. uniform variables

If

$$
X_1,\ldots,X_n
\stackrel{\mathrm{i.i.d.}}{\sim}
U(0,1),
$$

then

$$
F_M(x)=x^n,
$$

and

$$
f_M(x)=nx^{n-1},
\qquad
0<x<1.
$$

Thus

$$
\boxed{
\mathbb E[M]=\frac{n}{n+1}.
}
$$


In [ ]:
max_n = widgets.IntSlider(value=3, min=1, max=20, description="n")
max_output = widgets.Output()


def update_max_uniform(*_):
    with max_output:
        clear_output(wait=True)

        n = max_n.value
        x = np.linspace(0,1,700)
        f = n*x**(n-1)

        fig, ax = plt.subplots(figsize=(8,3.3))
        ax.plot(x,f)
        ax.set_xlabel("x")
        ax.set_ylabel("f_M(x)")
        ax.set_title("Density of the maximum of n U(0,1) variables")
        plt.show()

        display(Math(
            r"\mathbb E[M]=\frac{n}{n+1}="
            + f"{n/(n+1):.6f}"
        ))


max_n.observe(update_max_uniform, names="value")
display(widgets.VBox([max_n,max_output]))
update_max_uniform()


### Minimum of independent exponentials

If

$$
X_i\sim\operatorname{Exp}(\lambda_i)
$$

independently, then

$$
P(m>x)
=
\prod_i e^{-\lambda_ix}
=
e^{-(\lambda_1+\cdots+\lambda_n)x}.
$$

Therefore

$$
\boxed{
m
\sim
\operatorname{Exp}
(\lambda_1+\cdots+\lambda_n).
}
$$


## 17. Order statistics

For real random variables $X_1,\ldots,X_n$, let

$$
X_{(1)}
\le
X_{(2)}
\le
\cdots
\le
X_{(n)}
$$

be the ordered sample.

For an i.i.d. sample from a density $f$,

$$
\boxed{
f_{(1),\ldots,(n)}(x_1,\ldots,x_n)
=
n!\prod_{i=1}^n f(x_i),
\qquad
x_1<\cdots<x_n.
}
$$


### Density of the $k$-th order statistic

If the common cdf is $F$, then

$$
\boxed{
f_{X_{(k)}}(x)
=
\frac{n!}{(k-1)!(n-k)!}
F(x)^{k-1}
[1-F(x)]^{n-k}
f(x).
}
$$


### Median of three uniform variables

For $n=3$ and $k=2$ with $U(0,1)$ sampling,

$$
\boxed{
f_{X_{(2)}}(x)
=
6x(1-x),
\qquad
0<x<1.
}
$$


In [ ]:
x = np.linspace(0,1,700)
median_density = 6*x*(1-x)

fig, ax = plt.subplots(figsize=(8,3.3))
ax.plot(x,median_density)
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Median of three independent U(0,1) observations")
plt.show()

display(Math(
    r"\int_0^1 6x(1-x)\,dx=1"
))


## 18. Discrete convolution

If $X,Y$ are independent and integer-valued almost surely, then

$$
\boxed{
P(X+Y=n)
=
\sum_{k\in\mathbb Z}
P(X=k)P(Y=n-k).
}
$$

Independence enters through the factorization of each joint event.


### Sum of two fair dice

For independent fair dice $X,Y$,

$$
P(X+Y=n)
=
\begin{cases}
(n-1)/36,&2\le n\le7,\\
(13-n)/36,&8\le n\le12.
\end{cases}
$$


In [ ]:
sums = np.arange(2,13)
probs = np.array([
    sum(
        (1/6)*(1/6)
        for k in range(1,7)
        if 1 <= s-k <= 6
    )
    for s in sums
])

fig, ax = plt.subplots(figsize=(8,3.3))
ax.bar(sums,probs)
ax.set_xlabel("sum")
ax.set_ylabel("probability")
ax.set_title("Discrete convolution: sum of two fair dice")
plt.show()

display(Markdown(f"Total mass: **{probs.sum():.12f}**"))


## 19. Continuous convolution

If $X,Y$ are independent with densities $f_X,f_Y$, then

$$
\boxed{
f_{X+Y}(z)
=
\int_{-\infty}^{\infty}
f_X(x)f_Y(z-x)\,dx
}
$$

for almost every $z$.

The formula can be obtained by the Jacobian transformation

$$
U=X+Y,
\qquad
V=X,
$$

followed by marginalization.


### Sum of two independent $U(0,1)$ variables

The convolution density is

$$
\boxed{
f_Z(z)
=
\begin{cases}
z,&0<z<1,\\
2-z,&1\le z<2,\\
0,&\text{otherwise}.
\end{cases}
}
$$

The sum is not uniform.


In [ ]:
z = np.linspace(-0.2,2.2,1000)
fz = np.where(
    (0 < z) & (z < 1),
    z,
    np.where(
        (1 <= z) & (z < 2),
        2-z,
        0.0,
    ),
)

fig, ax = plt.subplots(figsize=(8,3.3))
ax.plot(z,fz)
ax.set_xlabel("z")
ax.set_ylabel("f_{X+Y}(z)")
ax.set_title("Triangular convolution density")
plt.show()

display(Math(
    r"P(X+Y\le1/2)=\int_0^{1/2}z\,dz=\frac18"
))


### Variance of independent sums

For mutually independent variables with finite variances,

$$
\boxed{
\operatorname{Var}
\left(
\sum_i X_i
\right)
=
\sum_i
\operatorname{Var}(X_i).
}
$$

The same variance identity requires only pairwise zero covariance.


## 20. The bivariate normal family

For $\sigma_X,\sigma_Y>0$ and $-1<\rho<1$, the bivariate normal density is

$$
\boxed{
\begin{aligned}
f_{X,Y}(x,y)
&=
\frac{
1
}{
2\pi\sigma_X\sigma_Y\sqrt{1-\rho^2}
}\\
&\quad\times
\exp
\left\{
-\frac{
z^2-2\rho zw+w^2
}{
2(1-\rho^2)
}
\right\},
\end{aligned}
}
$$

where

$$
z=\frac{x-\mu_X}{\sigma_X},
\qquad
w=\frac{y-\mu_Y}{\sigma_Y}.
$$


### Construction from independent standard normals

Let $Z_1,Z_2$ be independent $N(0,1)$ variables and define

$$
X
=
\mu_X+\sigma_XZ_1,
$$

$$
Y
=
\mu_Y
+
\sigma_Y
\left(
\rho Z_1
+
\sqrt{1-\rho^2}\,Z_2
\right).
$$

Then

$$
X\sim N(\mu_X,\sigma_X^2),
$$

$$
Y\sim N(\mu_Y,\sigma_Y^2),
$$

and

$$
\boxed{
\operatorname{Cov}(X,Y)
=
\rho\sigma_X\sigma_Y.
}
$$

Therefore the correlation coefficient is exactly $\rho$.


In [ ]:
bvn_rho = widgets.FloatSlider(value=0.6, min=-0.95, max=0.95, step=0.05, description="rho")
bvn_N = widgets.IntSlider(value=5000, min=500, max=30000, step=500, description="N")
bvn_output = widgets.Output()


def update_bivariate_normal(*_):
    with bvn_output:
        clear_output(wait=True)

        rho = bvn_rho.value
        N = bvn_N.value

        X,Y = bivariate_normal_sample(
            N,
            mu_x=1,
            mu_y=-1,
            sigma_x=2,
            sigma_y=3,
            rho=rho,
            seed=2026,
        )

        empirical_cov = np.cov(X,Y,ddof=0)[0,1]
        empirical_corr = np.corrcoef(X,Y)[0,1]

        display(Math(
            r"\widehat{\operatorname{Cov}}(X,Y)="
            + f"{empirical_cov:.6f}"
        ))
        display(Math(
            r"\text{theory}=\rho\sigma_X\sigma_Y="
            + f"{rho*2*3:.6f}"
        ))
        display(Math(
            r"\widehat\rho="
            + f"{empirical_corr:.6f}"
        ))

        fig, ax = plt.subplots(figsize=(5.5,5))
        ax.scatter(X,Y,s=4,alpha=0.2)
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_title("Bivariate normal sample")
        plt.show()


for control in (bvn_rho,bvn_N):
    control.observe(update_bivariate_normal, names="value")

display(widgets.VBox([
    widgets.HBox([bvn_rho,bvn_N]),
    bvn_output,
]))
update_bivariate_normal()


### Zero correlation and independence

Inside the non-degenerate bivariate normal family,

$$
\boxed{
X\text{ and }Y\text{ are independent}
\iff
\rho_{X,Y}=0.
}
$$

This is a special property of the bivariate normal family.

It does **not** contradict the earlier example $Y=X^2$, because that pair is not bivariate normal.


## 21. Historical problem: Galton's quincunx

An idealized Galton board has $n$ rows of pins.

At each row the particle makes an independent left--right choice with probability $1/2$.

Let

$$
B_i=
\mathbf 1_{\{\text{right move at row }i\}},
$$

and

$$
S_n=B_1+\cdots+B_n.
$$

Then

$$
\boxed{
S_n
\sim
\operatorname{Bin}
\left(
n,\frac12
\right).
}
$$

Thus

$$
P(S_n=k)
=
\binom nk2^{-n},
$$

$$
\mathbb E[S_n]
=
\frac n2,
$$

and

$$
\operatorname{Var}(S_n)
=
\frac n4.
$$


### Model assumption versus physical device

The mathematical model requires:

- left and right to have probability $1/2$;
- decisions at different rows to be independent.

A real board may violate these assumptions because of friction, imperfect pins or repeated collisions.

The physical device illustrates the model. It does not make the assumptions automatically true.


In [ ]:
galton_n = widgets.IntSlider(value=20, min=2, max=100, description="rows")
galton_N = widgets.IntSlider(value=30000, min=1000, max=100000, step=1000, description="particles")
galton_output = widgets.Output()


def update_galton(*_):
    with galton_output:
        clear_output(wait=True)

        n = galton_n.value
        N = galton_N.value

        rng = np.random.default_rng(2026)
        right_steps = rng.binomial(n,0.5,size=N)

        counts = np.bincount(right_steps,minlength=n+1)/N
        values = np.arange(n+1)
        exact = np.array([
            binomial_pmf(k,n,0.5)
            for k in values
        ])

        fig, ax = plt.subplots(figsize=(8,3.5))
        ax.bar(values,counts,alpha=0.4,label="simulation")
        ax.plot(values,exact,"o-",label="exact Bin(n,1/2)")
        ax.set_xlabel("number of right moves")
        ax.set_ylabel("probability")
        ax.set_title("Galton board and the binomial law")
        ax.legend()
        plt.show()

        display(Math(
            r"\text{sample mean}="
            + f"{right_steps.mean():.6f}"
        ))
        display(Math(
            r"\text{theoretical mean}="
            + f"{n/2:.6f}"
        ))
        display(Math(
            r"\text{sample variance}="
            + f"{right_steps.var():.6f}"
        ))
        display(Math(
            r"\text{theoretical variance}="
            + f"{n/4:.6f}"
        ))


for control in (galton_n,galton_N):
    control.observe(update_galton, names="value")

display(widgets.VBox([
    widgets.HBox([galton_n,galton_N]),
    galton_output,
]))
update_galton()


## 22. Python laboratory: dependence, covariance and sums

The following experiments emphasize three distinctions:

1. zero correlation does not imply independence;
2. convolution changes the shape of sums;
3. Galton aggregation turns many binary decisions into a binomial pattern.


In [ ]:
rng = np.random.default_rng(2026)
N = 50000

X = rng.uniform(-1,1,size=N)
Y = X**2

rho = np.corrcoef(X,Y)[0,1]

display(Math(
    r"\widehat\rho_{X,X^2}="
    + f"{rho:.6f}"
))

fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(X[:4000],Y[:4000],s=4,alpha=0.2)
ax.set_xlabel("X")
ax.set_ylabel("X^2")
ax.set_title("Strong nonlinear dependence with near-zero correlation")
plt.show()


## 23. Guided exercise generator


In [ ]:
exercise_rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random", "random"),
        ("Joint pmf", "pmf"),
        ("Marginal", "marginal"),
        ("Independence", "independence"),
        ("Covariance", "covariance"),
        ("Correlation", "correlation"),
        ("Multinomial", "multinomial"),
        ("Order statistic", "order"),
        ("Convolution", "convolution"),
        ("Bivariate normal", "bvn"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
prompt_output = widgets.Output()
feedback_output = widgets.Output()
state = {}


def make_exercise(_=None):
    kind = exercise_kind.value

    if kind == "random":
        kind = exercise_rng.choice([
            "pmf",
            "marginal",
            "independence",
            "covariance",
            "correlation",
            "multinomial",
            "order",
            "convolution",
            "bvn",
        ])

    if kind == "pmf":
        target = "1"
        prompt = "What must the entries of a finite joint pmf sum to?"
        hint = "The joint pmf is a probability law on pairs."
        solution = r"\sum_{x,y}p_{X,Y}(x,y)=1."

    elif kind == "marginal":
        target = "0.6666666667"
        prompt = "In the chapter's dependent table, find P(X=1)."
        hint = "Sum the row corresponding to X=1."
        solution = r"P(X=1)=\frac23."

    elif kind == "independence":
        target = "no"
        prompt = "Does pairwise independence of three variables imply mutual independence? yes/no"
        hint = "Recall X=U, Y=V, Z=(U+V) mod 2."
        solution = r"\text{No.}"

    elif kind == "covariance":
        target = "121"
        prompt = "Var(X)=4, Var(Y)=9, Cov(X,Y)=-2. Find Var(2X-3Y)."
        hint = "Use the bilinear variance formula."
        solution = r"\operatorname{Var}(2X-3Y)=121."

    elif kind == "correlation":
        target = "no"
        prompt = "Does Cov(X,Y)=0 imply independence for arbitrary variables? yes/no"
        hint = "Use Y=X^2 as a counterexample."
        solution = r"\text{No.}"

    elif kind == "multinomial":
        target = "-0.3333333333"
        prompt = "For 12 fair die rolls, find Cov(N_1,N_2)."
        hint = "Use -n p_i p_j."
        solution = r"\operatorname{Cov}(N_1,N_2)=-\frac13."

    elif kind == "order":
        target = "0.75"
        prompt = "For three iid U(0,1) variables, find E[max]."
        hint = "Use n/(n+1)."
        solution = r"\mathbb E[M]=\frac34."

    elif kind == "convolution":
        target = "0.125"
        prompt = "For independent U(0,1) variables, find P(X+Y<=1/2)."
        hint = "The density equals z on (0,1)."
        solution = r"P(X+Y\le1/2)=\frac18."

    else:
        target = "yes"
        prompt = "For a non-degenerate bivariate normal pair, does zero covariance imply independence? yes/no"
        hint = "This is a special property of the bivariate normal family."
        solution = r"\text{Yes.}"

    state.clear()
    state.update(
        target=target,
        hint=hint,
        solution=solution,
    )

    answer_box.value = ""

    with prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))

    with feedback_output:
        clear_output(wait=True)


def show_hint(_):
    with feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + state["hint"]))


def reveal(_):
    with feedback_output:
        clear_output(wait=True)
        display(Math(state["solution"]))


def check(_):
    with feedback_output:
        clear_output(wait=True)

        guess = answer_box.value.strip().lower().replace(" ","")
        target = state["target"].replace(" ","")

        correct = guess == target

        if not correct:
            try:
                correct = abs(float(guess)-float(target)) < 5e-4
            except Exception:
                pass

        display(Markdown(
            "**Correct.**"
            if correct
            else "**Not yet. Identify the joint-law structure before computing.**"
        ))


new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal)
check_button.on_click(check)

display(widgets.VBox([
    widgets.HBox([exercise_kind,new_button]),
    prompt_output,
    widgets.HBox([answer_box,check_button]),
    widgets.HBox([hint_button,reveal_button]),
    feedback_output,
]))

make_exercise()


## 24. AI Audit: joint distributions and dependence

Audit every AI-generated argument using the following checklist.

1. Does a proposed joint pmf sum to one?
2. Are marginals obtained by summing out the omitted coordinate?
3. Are marginals being incorrectly treated as sufficient to reconstruct the joint law?
4. Does a proposed joint density integrate to one?
5. Are marginal densities obtained by integrating out the other coordinate?
6. Is a joint density value being confused with a point probability?
7. Is a planar null set incorrectly assigned positive probability under a genuine planar density?
8. Is the diagonal example $Y=X$ recognized as having no joint density despite both marginals having densities?
9. Is multivariate LOTUS applied to the actual joint law?
10. Is pairwise independence being confused with mutual independence?
11. Is the cdf factorization criterion being used correctly?
12. Is pmf or density factorization being used in both directions only under the appropriate hypotheses?
13. Is independence of separate measurable transformations justified through inverse images?
14. Is expectation factorization presented as an **if and only if** characterization for bounded Borel transforms?
15. In the expectation-factorization converse, are indicator functions used?
16. Is the Jacobian of the **inverse** transformation used in the density formula?
17. Is the transformed support computed correctly?
18. Is covariance computed as $\mathbb E[XY]-\mathbb E[X]\mathbb E[Y]$?
19. Is zero covariance being confused with independence?
20. Is correlation constrained to $[-1,1]$?
21. Is $|\rho|=1$ correctly interpreted as an affine relation almost surely?
22. Is a covariance matrix recognized as symmetric positive semidefinite?
23. Are multinomial off-diagonal covariances negative?
24. Is a maximum cdf computed by intersecting coordinate events?
25. Does an order-statistic density include the correct combinatorial factor?
26. Is independence used where convolution requires it?
27. Is the sum of two $U(0,1)$ variables incorrectly called uniform?
28. Is zero correlation used to imply independence only in the non-degenerate bivariate normal family?
29. Does the Galton-board model explicitly assume independent fair left--right decisions?
30. Is simulation being used as illustration rather than proof?

### Claims to audit

- “The marginals uniquely determine the joint law.”
- “Pairwise independence implies mutual independence.”
- “Zero covariance always implies independence.”
- “The sum of two independent $U(0,1)$ variables is $U(0,2)$.”
- “Any two variables with normal marginals and zero covariance must be independent.”

All five are false as written.


### Suggested AI-guided activities

- “Give me a small joint pmf with one unknown normalizing constant. Ask me to normalize it, compute both marginals, test independence and then compute $\mathbb E[XY]$.”
- “Contrast independent variables, positively correlated variables and dependent variables with zero correlation.”
- “Give me three pairwise independent Bernoulli-derived variables that fail mutual independence and make me locate the failed triple event.”
- “Guide me through the expectation-factorization characterization starting from indicators and simple functions.”
- “Give me a two-dimensional transformation and require me to find the inverse map, transformed support and inverse Jacobian before writing the density.”
- “Generate a covariance matrix and ask me to test positive semidefiniteness through quadratic forms and eigenvalues.”
- “Give me a multinomial count vector and make me explain the negative off-diagonal covariances.”
- “Guide me through Galton's quincunx without first naming the binomial law.”


## 25. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. Marginals generally determine the full joint law:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{Dependence information is invisible in the marginals alone.}",
    ),
    (
        "2. A discrete marginal pmf is obtained by:",
        ["Choose...", "summing out the other coordinate", "multiplying row entries"],
        "summing out the other coordinate",
        r"p_X(x)=\sum_y p_{X,Y}(x,y).",
    ),
    (
        "3. If X and Y each have a density, the pair must have a joint density:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{The example }Y=X\text{ is singular on the diagonal.}",
    ),
    (
        "4. Pairwise independence implies mutual independence:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{The parity construction is a counterexample.}",
    ),
    (
        "5. Expectation factorization for all bounded Borel transforms characterizes independence:",
        ["Choose...", "true", "false"],
        "true",
        r"\text{Indicators give the converse direction.}",
    ),
    (
        "6. Separate measurable transformations preserve independence:",
        ["Choose...", "true", "false"],
        "true",
        r"\text{Use Borel inverse images.}",
    ),
    (
        "7. Independence with finite second moments implies zero covariance:",
        ["Choose...", "true", "false"],
        "true",
        r"\mathbb E[XY]=\mathbb E[X]\mathbb E[Y].",
    ),
    (
        "8. Zero covariance implies independence for arbitrary variables:",
        ["Choose...", "true", "false"],
        "false",
        r"Y=X^2\text{ provides a counterexample.}",
    ),
    (
        "9. Every covariance matrix is positive semidefinite:",
        ["Choose...", "true", "false"],
        "true",
        r"\mathbf a^\top\Sigma\mathbf a=\operatorname{Var}(\mathbf a^\top\mathbf X)\ge0.",
    ),
    (
        "10. Multinomial counts in different categories have:",
        ["Choose...", "negative covariance", "positive covariance", "zero covariance always"],
        "negative covariance",
        r"\operatorname{Cov}(N_i,N_j)=-np_ip_j.",
    ),
    (
        "11. The sum of two independent U(0,1) variables is uniform on (0,2):",
        ["Choose...", "true", "false"],
        "false",
        r"\text{Its density is triangular.}",
    ),
    (
        "12. In the non-degenerate bivariate normal family, rho=0 implies independence:",
        ["Choose...", "true", "false"],
        "true",
        r"\rho=0\text{ makes the joint density factorize.}",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    dropdown = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="470px"),
    )

    quiz_widgets.append(dropdown)

    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:670px'>{prompt}</div>"),
        dropdown,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)

        score = sum(
            widget.value == correct
            for widget, (_,_,correct,_) in zip(
                quiz_widgets,
                quiz_data,
            )
        )

        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))

        for i, (
            widget,
            (_,_,correct,explanation),
        ) in enumerate(zip(quiz_widgets,quiz_data),1):

            mark = "✓" if widget.value == correct else "✗"

            display(Markdown(
                f"**{mark} Question {i}:** correct answer = `{correct}`"
            ))

            display(Math(explanation))


grade_button.on_click(grade_quiz)

display(widgets.VBox(
    quiz_rows + [grade_button, quiz_output]
))


## 26. Automatic mathematical verification

The final cell checks representative identities from the chapter.


In [ ]:
# Dependent discrete table normalization and marginals.
table = np.array([
    [1/6,1/6,0],
    [0,1/3,1/3],
])

assert np.isclose(table.sum(),1)

px,py = joint_pmf_marginals(table)

assert np.allclose(px,[1/3,2/3])
assert np.allclose(py,[1/6,1/2,1/3])
assert not np.allclose(table,np.outer(px,py))

# Triangle moments and covariance.
EX = 2/3
EY = 1/3
EXY = 1/4

assert abs(EXY-EX*EY-1/36) < 1e-12

# Pairwise-but-not-mutual construction.
triples = [
    (u,v,(u+v)%2)
    for u,v in [(0,0),(0,1),(1,0),(1,1)]
]

assert sum(0.25 for t in triples if t==(0,0,0)) == 0.25
assert 0.5*0.5*0.5 == 0.125

# Expectation factorization under an independent product law.
x_vals = np.array([0,1])
px = np.array([0.4,0.6])

y_vals = np.array([1,3])
py = np.array([0.7,0.3])

g = lambda x: 2*x+1
h = lambda y: y*y

Eg = np.sum(np.array([g(x) for x in x_vals])*px)
Eh = np.sum(np.array([h(y) for y in y_vals])*py)

Egh = sum(
    g(x)*h(y)*px[i]*py[j]
    for i,x in enumerate(x_vals)
    for j,y in enumerate(y_vals)
)

assert abs(Egh-Eg*Eh) < 1e-12

# Variance of linear combination.
assert 4*4 + 9*9 + 2*2*(-3)*(-2) == 121

# Correlation counterexample X in {-1,0,1}, Y=X^2.
x = np.array([-1,0,1],dtype=float)
p = np.array([1/3,1/3,1/3])
y = x*x

EX = np.sum(x*p)
EY = np.sum(y*p)
EXY = np.sum(x*y*p)

assert abs(EXY-EX*EY) < 1e-12

# Triangle covariance matrix PSD.
Sigma = np.array([
    [1/18,1/36],
    [1/36,1/18],
])

assert np.all(np.linalg.eigvalsh(Sigma) >= -1e-12)

# Multinomial covariance for 12 fair die rolls.
assert abs(-12*(1/6)*(1/6) + 1/3) < 1e-12

# Maximum of three uniforms.
assert abs(3/4 - 0.75) < 1e-12

# Order-statistic median density normalization.
grid = np.linspace(0,1,50001)
median_density = 6*grid*(1-grid)

assert abs(numerical_integral_1d(median_density,grid)-1) < 1e-9

# Two-dice convolution normalization.
sums = np.arange(2,13)
dice_probs = np.array([
    sum(
        (1/6)*(1/6)
        for k in range(1,7)
        if 1 <= s-k <= 6
    )
    for s in sums
])

assert abs(dice_probs.sum()-1) < 1e-12

# U(0,1)+U(0,1) density normalization and probability.
z = np.linspace(0,2,50001)
fz = np.where(z<1,z,2-z)

assert abs(numerical_integral_1d(fz,z)-1) < 1e-9

mask = z <= 0.5
assert abs(numerical_integral_1d(fz[mask],z[mask])-1/8) < 1e-8

# Bivariate normal construction covariance.
rho = 0.6
sigma_x = 2
sigma_y = 3

assert abs(rho*sigma_x*sigma_y - 3.6) < 1e-12

# Galton formulas.
n = 10

assert abs(sum(binomial_pmf(k,n,0.5) for k in range(n+1))-1) < 1e-12
assert n/2 == 5
assert n/4 == 2.5

show_result(
    "All Chapter 11 automatic checks passed",
    r"p_X(x)=\sum_y p_{X,Y}(x,y)",
    r"\operatorname{Cov}(X,Y)=\mathbb E[XY]-\mathbb E[X]\mathbb E[Y]",
    r"\mathbf a^\top\Sigma\mathbf a\ge0",
    r"\operatorname{Cov}(N_i,N_j)=-np_ip_j",
    r"f_{X+Y}(z)=\int f_X(x)f_Y(z-x)\,dx",
    r"\rho=0\Longleftrightarrow\text{ independence in the non-degenerate bivariate normal family}",
    note=(
        "Marginalization, covariance, PSD, multinomial, order-statistic, "
        "convolution and bivariate-normal checks all passed."
    ),
)


## 27. Chapter map

| Chapter concept | Computational representation |
|---|---|
| random vector | coordinate collection on one probability space |
| joint cdf | rectangle increments |
| marginal cdf | unrestricted-coordinate limit |
| joint pmf | finite probability table |
| marginal pmf | row and column sums |
| marginals do not determine joint law | two different tables with same marginals |
| joint density | region integration |
| marginal density | integrate out one coordinate |
| singular pair | $Y=X$ diagonal example |
| multivariate LOTUS | mixed moment on triangular support |
| mutual independence | Borel-set factorization |
| pairwise vs mutual | parity construction |
| cdf factorization | independence criterion |
| pmf/density factorization | discrete and continuous criteria |
| measurable transformations | preservation of independence |
| expectation factorization | if-and-only-if independence characterization |
| Jacobian transformation | stretched square and sum/difference transform |
| covariance | mixed-moment formula |
| bilinearity | variance of a linear combination |
| uncorrelated dependence | $Y=X^2$ |
| correlation | Cauchy--Schwarz bound and affine equality case |
| covariance matrix | quadratic form and eigenvalue check |
| multinomial | negative off-diagonal covariances |
| maximum/minimum | cdf products |
| order statistics | $n!$ joint density and $k$-th formula |
| discrete convolution | sum of two dice |
| density convolution | sum of two uniforms |
| bivariate normal | construction from independent standard normals |
| normal zero correlation | independence iff $\rho=0$ |
| Galton quincunx | independent Bernoulli aggregation |
| AI Audit | structural checks on dependence claims |

The central hierarchy is:

$$
\boxed{
\text{joint law}
\longrightarrow
\text{dependence structure}
\longrightarrow
\text{covariance/correlation summaries}
}
$$

and not the reverse: covariance alone does not determine the joint law.
